In [1]:
from pathlib import Path
import json
import shutil

# Notebook is in notebooks/02_preprocess -> go up 2 levels
RAW_ROOT = Path("../../data/raw/archive (4)")
OUT_ROOT = Path("../../data/processed/archive4_via")

print("RAW_ROOT:", RAW_ROOT.resolve())
print("RAW_ROOT exists:", RAW_ROOT.exists())

# 1) Auto-find JSON annotation file(s)
json_files = list(RAW_ROOT.rglob("*.json"))
print("\nJSON files found:")
for j in json_files:
    print("-", j.relative_to(RAW_ROOT))

# 2) Auto-find image files
img_exts = (".jpg",".jpeg",".png",".JPG",".JPEG",".PNG")
img_files = [p for p in RAW_ROOT.rglob("*") if p.suffix in img_exts]
print("\nTotal images found:", len(img_files))

# If nothing found, stop early
if len(img_files) == 0:
    raise FileNotFoundError("No images found inside archive (4). Check the folder content.")

# Choose the first json as annotation (if multiple, we’ll handle after)
ann_path = json_files[0] if len(json_files) > 0 else None
print("\nUsing annotation JSON:", ann_path.relative_to(RAW_ROOT) if ann_path else "None (no json found)")

# 3) Try to read JSON and count boxes/classes (works for common formats)
total_boxes = 0
class_set = set()

if ann_path:
    data = json.loads(ann_path.read_text(encoding="utf-8"))

    # COCO style: {"images":[], "annotations":[], "categories":[]}
    if isinstance(data, dict) and "annotations" in data:
        anns = data.get("annotations", [])
        cats = data.get("categories", [])
        total_boxes = len(anns)
        for c in cats:
            if "name" in c:
                class_set.add(c["name"])
        print("\nCOCO-like JSON detected \n")
        print("Images in JSON:", len(data.get("images", [])))
        print("Annotations (boxes):", total_boxes)
        print("Classes:", sorted(list(class_set))[:20], ("..." if len(class_set) > 20 else ""))

    # VIA style often: dict of entries where each entry has "regions"
    elif isinstance(data, dict):
        print("\nVIA-like JSON detected  (dictionary entries)")
        for k, v in data.items():
            regions = v.get("regions", [])
            # regions might be list or dict
            if isinstance(regions, dict):
                regions = list(regions.values())
            for r in regions:
                total_boxes += 1
                attrs = r.get("region_attributes", {})
                # try common label keys
                for key in ["label", "class", "damage", "type"]:
                    if key in attrs:
                        class_set.add(str(attrs[key]))
        print("Total boxes (regions):", total_boxes)
        print("Classes (from region_attributes):", sorted(list(class_set))[:20], ("..." if len(class_set) > 20 else ""))

    else:
        print("\nUnknown JSON structure. We will still copy it as-is.")

# 4) PREPROCESS (safe): create clean output + copy images and JSON (no resizing)
OUT_IMG_DIR = OUT_ROOT / "images"
OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

# Copy images
copied = 0
for p in img_files:
    dst = OUT_IMG_DIR / p.name
    if not dst.exists():
        shutil.copy2(p, dst)
        copied += 1

# Copy JSON
if ann_path:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ann_path, OUT_ROOT / ann_path.name)

print("\n PREPROCESS DONE")
print("Copied images:", copied)
print("Saved to:", OUT_ROOT.resolve())
if ann_path:
    print("Copied annotation JSON:", (OUT_ROOT / ann_path.name).name)

# 5) Quick verify output
out_imgs = list(OUT_IMG_DIR.glob("*"))
print("\nVERIFY OUTPUT:")
print("Processed images count:", len(out_imgs))
print("Annotation boxes counted (if detectable):", total_boxes)
print("Class count (if detectable):", len(class_set))


RAW_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (4)
RAW_ROOT exists: True

JSON files found:
- 0Train_via_annos.json
- 0Val_via_annos.json

Total images found: 13945

Using annotation JSON: 0Train_via_annos.json

VIA-like JSON detected  (dictionary entries)
Total boxes (regions): 30046
Classes (from region_attributes): [] 

 PREPROCESS DONE
Copied images: 0
Saved to: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_via
Copied annotation JSON: 0Train_via_annos.json

VERIFY OUTPUT:
Processed images count: 13945
Annotation boxes counted (if detectable): 30046
Class count (if detectable): 0
